# Plot Az Scalar-Amplitude Synth Recovery — Nicoya

Counterpart to `plt_test_syn_slip_az_inv_hetmu_nicoyaCK_locking.ipynb`.

**Old scheme**: invert (s_strike, s_dip) 2-component slip; compare ||s_rec|| vs ||s_true||  
**New scheme**: invert scalar amplitude a(x) ∈ [0, amp_max] with fixed plate-convergence direction;
compare a_rec(x) vs a_true(x) and the recovery ratio a_rec/a_true.

The amplitude represents the back-slip scaling factor:
- a = 80 mm/yr → fully locked
- a = 0 → freely creeping

Key output files from `synth_stripeslip_az_scalaramp_inv_hetmu_uneven_nicoyaCK_lock_noi2.py`:
- `slip_recovery_<meshname><slip_str><mu_for><inv_str><mu_inv>.txt` — per-vertex recovery table
- `m_s_fault_<...>.txt` — 2-column (s_strike, s_dip) for backward compat (all γ values)

In [ ]:
import os
import numpy as np
import pandas as pd
import pygmt
import matplotlib
import sys
import meshio

# Single-threaded to avoid conflicts with FEniCS MPI
for var in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
            'VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[var] = '1'

sys.path.insert(0, '/home/staff/chao/SSEinv/Nicoya/codes')
import utils_plot as utp

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
datadir    = "/home/staff/chao/SSEinv/Nicoya/"
figpath    = "/home/staff/chao/SSEinv/Nicoya/figures/synth/"
resultpath = "/home/staff/chao/SSEinv/Nicoya/syn_slip/"
os.makedirs(figpath, exist_ok=True)

# ── Mesh / analysis config ───────────────────────────────────────────────────
meshname     = "nicoyaCKden_une_sm"
pollute      = True
pollute_type = 'uniform'

# ── Unit conversion ──────────────────────────────────────────────────────────
m2mm = 1e3
km2m = 1e3

# ── GNSS stations ────────────────────────────────────────────────────────────
# read in the lon and lat of stations
# obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

# note that the height is in m, Dt in years, the original displacement data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                    'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                    'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive
# Convert mm to m, needed for inversion
cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
df[cols_to_convert] = df[cols_to_convert] / m2mm  # Convert mm to m

# displacement noise standard deviations, in m 
if pollute:
        if pollute_type == 'uniform':
                noise_std_h = 0.5 * (df['vx_std_Car'].mean() + df['vy_std_Car'].mean()) 
                noise_std_v = df['vz_std_Car'].mean()
                error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_v
                
        elif pollute_type == 'datastd':
                error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']
                
else:
        error_e, error_n, error_v = df['vx_std_Car']*0, df['vy_std_Car']*0, df['vz_std_Car']*0

w_h = int(1/noise_std_h)
w_v = int(1/noise_std_v)
print(f"w_h={w_h}, w_v={w_v}")

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
# print(df_plate.head())

# ── Trench and plotting region ───────────────────────────────────────────────
# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
# print(trench.head())

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
region1=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 


In [ ]:
# ── Volcano locations ─────────────────────────────────────────────────────────
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volc = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
volc = volc[(volc['lon'] > -87) & (volc['lon'] < -84) &
            (volc['lat'] >  8.5) & (volc['lat'] < 11.5)]

In [ ]:
### Choose the slip model, all models are in m_dip direction, and m_strike = 0
### All are offseted a little along dip due to the plate shape from CK
# slipmodel = 1     # along-strike 3-stripe pattern, shallow-middle-deep
slipmodel = 2     # along-strike 2-stripe pattern with gap, shallow-deep 
# slipmodel = 3     # along-strike 1-stripe pattern, complementary of model 2, middle
# slipmodel = 4     # checkerboard pattern in strike-dip direction
# slipmodel = 5     # checkerboard pattern in x and y direction  
# slipmodel = 6     # same as model 4, but amp up to 1 m rather than V_norm=78.5 mm
# slipmodel = 7     # 2D Gaussian where the contours are circular, within range of 0 and max

# slip_azimuth_deg  = 45.0   # CW from North; N45E = Cocos-Caribbean trench-normal convergence
# slip_azimuth_deg  = 26.0   # CW from North; N26E = roughly is oblique convergence
slip_azimuth_deg  = 33.5   # CW from North; N33.5E, a little oblique convergence

# Define true model: slip in fixed geographic azimuth, magnitude = checkerboard pattern
V_norm = 78.5   # in mm
V_para = 27
V_para_const = 11

if slip_azimuth_deg == 45.0:
    amp = V_norm  # 78.5 mm
elif slip_azimuth_deg == 26.0:
    amp = round(np.sqrt(V_norm**2+V_para**2))  # about 83 mm
elif slip_azimuth_deg == 33.5:
    amp = round(np.sqrt(V_norm**2+(V_para-V_para_const)**2))  # about 80 mm

amp = amp / 1e3     # convert mm to m

if slipmodel == 2:

    ### define the pattern of the slip distribution
    slip_pattern = "stripe"
    ### stripe pattern options
    pattern_option = 1  # 1: 2-stripe, shallow-deep; 2: 1-stripe, intermediate

    # Stripe pattern in m_dip, along-strike, 2-stripe, shallow-deep; m_strike = 0
    # amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 80e3     # width of each rectangle in dip direction
    y_len = 300e3   # length of each rectangle in strike direction  
    dx = 35e3  # gap between rectangles
    stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
    x0 = 0     # x center of the pattern
    y0 = -45e3     # y center of the pattern
    pattern_rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)

    # y_len = 240e3   # length of each rectangle in strike direction  
    # dx = 40e3  # gap between rectangles
    # y0 = -40e3     # y center of the pattern
    # zmin = -70e3
    # zmax = 0

    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

elif slipmodel == 4:

    ### define the pattern of the slip distribution
    slip_pattern = "checker"
    ### checkerboard pattern options
    pattern_option = 1  # 1: along-strike/dip directions; 2: along N-E directions

    # Checkerboard pattern in m_dip, in strike-dip direction; m_strike = 0
    # amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 35e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 35e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = -20e3  # offset along strike
    y0 = -60e3  # offset along dip
    pattern_rot_deg = 45  # 45° counterclockwise
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    # slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{pattern_rot_deg:g}_ms{amp:g}"    
    slip_str_gt = (
        f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

slip_str_gt1 = slip_str_gt  #with no pollution

if pollute:
    if pollute_type == 'uniform':
        slip_str_gt = slip_str_gt + "_pou"
    
    elif pollute_type == 'datastd':
        slip_str_gt = slip_str_gt + "_pod"

print("Slip model string:", slip_str_gt)    

In [ ]:
# ── Shear modulus identifier strings ─────────────────────────────────────────
homo_str    = "_mul40u40"           # homogeneous 40/40 GPa
sw_str      = "_mul55u25"           # slab-wedge 55/25 GPa
het3d_str   = "_DeShon3D_ref1D_2_hull_CG2"   # 3D DeShon, contrast=2

# ── Consistent-forward flag ───────────────────────────────────────────────────
# True  → load results from *_cf_* files (PDEVarf_AzSlip_Direct, no operator mismatch)
# False → load results from original mismatched-forward files
flag_cf = True

In [ ]:
# ── Mesh reference frame ─────────────────────────────────────────────────────
lon0, lat0 = -84, 7
rot  = 45          # degrees CCW
x0, y0 = 130e3, 350e3   # m (mesh offset)

# ── Load fault geometry ───────────────────────────────────────────────────────
fg = np.loadtxt(resultpath + 'fault_geometry_' + meshname + '.txt')   # m
xf, yf, zf = fg[::3,0], fg[::3,1], fg[::3,2]
xf_km, yf_km, zf_km = xf/km2m, yf/km2m, zf/km2m                      # km

# Convert fault vertex coords to lon/lat
lons_f, lats_f = utp.ckm2LLd(xf_km*km2m + x0, yf_km*km2m + y0, lon0, lat0, -rot)

# ── True amplitude ─────────────────────────────────────────────────────────────
if flag_cf:
    # Consistent-forward: true amplitude = m_phys (scalar CG1 param prescribed directly)
    # slip_true_cf_*: xf yf zf m_phys s_strike s_dip slip_mag c_mag
    slip_cf_raw = np.loadtxt(resultpath + 'slip_true_cf_' + meshname + slip_str_gt + '.txt',
                             comments='#')
    true_amp    = slip_cf_raw[:, 3]   # m_phys in m/yr
else:
    # Original: true amplitude = ||slip_vector|| from global-frame 2-component slip
    mtrue_s  = np.loadtxt(resultpath + 'mtrue_s_fault_' + meshname + slip_str_gt + '.txt')
    true_amp = np.sqrt(mtrue_s[:,0]**2 + mtrue_s[:,1]**2)   # m/yr

true_amp_mm = true_amp * m2mm   # mm/yr

# Build base fault DataFrame
df_fault = pd.DataFrame({
    'lon': lons_f, 'lat': lats_f,
    'xf': xf, 'yf': yf, 'zf': zf,
    'true_amp': true_amp, 'true_amp_mm': true_amp_mm
})
print(f"N fault vertices: {len(df_fault)}")
print(f"True amplitude range: [{true_amp_mm.min():.1f}, {true_amp_mm.max():.1f}] mm/yr")
print(f"Stripe fraction (>1 mm/yr): {(true_amp_mm > 1.0).mean()*100:.1f}%")

In [ ]:
slipmodel

In [ ]:
# ── Regularization parameters + inv_str for scalar amplitude scheme ───────────
rho_s   = 1e9

# Select γ to visualise — must match a completed inversion run
# Consistent-forward runs used γ=1.5e2; original runs used γ=1e2 (slipmodel==2)
if flag_cf:
    gamma_s = 1.5e2
else:
    if slipmodel == 2:
        gamma_s = 1e2
    elif slipmodel == 4:
        gamma_s = 5e1

delta_s = gamma_s / rho_s

if flag_cf:
    inv_str = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_s:.1e}_ds{delta_s:.1e}"
else:
    inv_str = f"_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_s:.1e}_ds{delta_s:.1e}"

print("inv_str:", inv_str)

# Forward / inversion shear modulus
mu_str_for = homo_str
mu_str_inv = homo_str
print("mu_str_for:", mu_str_for, "  mu_str_inv:", mu_str_inv)

In [ ]:
# ── Helper: load slip_recovery file (has per-vertex true/rec amp + ratio) ─────
def read_az_recovery(mu_str_for, inv_str, mu_str_inv):
    """
    Load slip_recovery_*.txt produced by solveCoseismicInversion_AzSlip.
    Two formats are handled:
      6-col (older scripts): xf yf zf true_amp rec_amp ratio_local
      7-col (newer scripts): xf yf zf true_amp rec_amp ratio_global ratio_local
    Coords are in LOCAL mesh metres (offset already subtracted).
    Returns DataFrame with lon/lat added, mm/yr columns, and 'ratio' = local ratio.
    """
    fname = (resultpath + 'slip_recovery_' + meshname + slip_str_gt
             + mu_str_for + inv_str + mu_str_inv + '.txt')
    raw = pd.read_csv(fname, sep=r'\s+', comment='#', header=None)
    if raw.shape[1] == 7:
        raw.columns = ['xf','yf','zf','true_amp','rec_amp','ratio_global','ratio_local']
        d = raw.copy()
        d['ratio'] = d['ratio_global']   # global: rec_amp / amp (no NaN)
        # d['ratio'] = d['ratio_local']       # local:  rec_amp / true_amp (NaN on creeping)
    else:
        # 6-col: ratio column is already local ratio
        raw.columns = ['xf','yf','zf','true_amp','rec_amp','ratio']
        d = raw.copy()
    # slip_recovery coords are in local m (x0/y0 already subtracted), add back for ckm2LLd
    lons, lats = utp.ckm2LLd(d['xf'] + x0, d['yf'] + y0, lon0, lat0, -rot)
    d['lon'] = lons; d['lat'] = lats
    d['true_amp_mm'] = d['true_amp'] * m2mm
    d['rec_amp_mm']  = d['rec_amp']  * m2mm
    return d


# ── Helper: load m_s_fault (all γ, backward compat) → amplitude ───────────────
def read_m_s_fault_amp(mu_str_for, inv_str, mu_str_inv):
    """
    Load m_s_fault_*.txt (2-col: s_strike, s_dip in m/yr) and compute amplitude.
    Uses df_fault for coordinates and true_amp for ratio.
    'ratio' = local recovery: rec_amp / true_amp (NaN on creeping patches).
    """
    fname = (resultpath + 'm_s_fault_' + meshname + slip_str_gt
             + mu_str_for + inv_str + mu_str_inv + '.txt')
    ms = np.loadtxt(fname)
    rec_amp = np.sqrt(ms[:,0]**2 + ms[:,1]**2)     # m/yr
    eps = 0.01 * amp
    ratio_local = np.where(true_amp > eps, rec_amp / np.where(true_amp > eps, true_amp, 1.0), np.nan)
    ratio_global = rec_amp / amp   # global recovery: divides by full plate velocity (no NaN)
    d = df_fault[['lon','lat','xf','yf','zf','true_amp','true_amp_mm']].copy()
    d['rec_amp']     = rec_amp
    d['rec_amp_mm']  = rec_amp * m2mm
    d['ratio']       = ratio_global
    d['s_strike_rec'] = ms[:,0]
    d['s_dip_rec']    = ms[:,1]
    return d

In [ ]:
# ── Load recovery data for selected γ ─────────────────────────────────────────
# Use slip_recovery if available (single-test γ=1e3); otherwise fall back to m_s_fault
import os
rec_file = (resultpath + 'slip_recovery_' + meshname + slip_str_gt
            + mu_str_for + inv_str + mu_str_inv + '.txt')
if os.path.exists(rec_file):
    df_rec = read_az_recovery(mu_str_for, inv_str, mu_str_inv)
    print("Loaded from slip_recovery file")
else:
    df_rec = read_m_s_fault_amp(mu_str_for, inv_str, mu_str_inv)
    print("slip_recovery not found — loaded from m_s_fault")

stripe_mask = df_rec['true_amp_mm'] > 1.0   # stripe vertices only
# stripe_mask = df_rec['true_amp_mm'] >= 0.0   # all
print(f"N vertices: {len(df_rec)}  |  stripe: {stripe_mask.sum()}")
print(f"True amp (stripe):   [{df_rec.loc[stripe_mask,'true_amp_mm'].min():.1f}, "
      f"{df_rec.loc[stripe_mask,'true_amp_mm'].max():.1f}] mm/yr")
print(f"Rec  amp (stripe):   [{df_rec.loc[stripe_mask,'rec_amp_mm'].min():.1f}, "
      f"{df_rec.loc[stripe_mask,'rec_amp_mm'].max():.1f}] mm/yr")
print(f"Recovery ratio (stripe): [{df_rec.loc[stripe_mask,'ratio'].min():.3f}, "
      f"{df_rec.loc[stripe_mask,'ratio'].max():.3f}]")

In [ ]:
# ── Load multiple γ values for L-curve recovery comparison ───────────────────
if slipmodel == 2:
    gammas_available = [1e1, 5e1, 1e2, 1.5e2, 2e2, 4e2, 6e2, 8e2, 1e3]
elif slipmodel == 4:
    gammas_available = [5e1, 1e2, 2e2, 4e2, 8e2, 1e3]

# For flag_cf: use read_az_recovery (slip_recovery_*_cf_*, stores m_phys directly)
# For original: use read_m_s_fault_amp (m_s_fault_*, stores m_phys*c_s/c_d → rec_amp=m_phys*||c||)
recs = {}
for gs in gammas_available:
    ds = gs / rho_s
    if flag_cf:
        istr = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gs:.1e}_ds{ds:.1e}"
        try:
            recs[gs] = read_az_recovery(mu_str_for, istr, mu_str_inv)
            d = recs[gs]
            sm = d['true_amp_mm'] > 1.0
            print(f"γ={gs:.1e}: rec amp [{d.loc[sm,'rec_amp_mm'].min():.1f}, "
                  f"{d.loc[sm,'rec_amp_mm'].max():.1f}] mm/yr  "
                  f"ratio [{d.loc[sm,'ratio'].min():.3f}, {d.loc[sm,'ratio'].max():.3f}]")
        except FileNotFoundError:
            print(f"γ={gs:.1e}: file not found, skipping")
    else:
        istr = f"_synscaamp_azbd_w{w_h}{w_v}_gs{gs:.1e}_ds{ds:.1e}"
        try:
            recs[gs] = read_m_s_fault_amp(mu_str_for, istr, mu_str_inv)
            d = recs[gs]
            sm = d['true_amp_mm'] > 1.0
            print(f"γ={gs:.1e}: rec amp [{d.loc[sm,'rec_amp_mm'].min():.1f}, "
                  f"{d.loc[sm,'rec_amp_mm'].max():.1f}] mm/yr  "
                  f"ratio [{d.loc[sm,'ratio'].min():.3f}, {d.loc[sm,'ratio'].max():.3f}]")
        except FileNotFoundError:
            print(f"γ={gs:.1e}: file not found, skipping")

In [ ]:
# ── Plot: 3-panel amplitude recovery (true | recovered | ratio) ───────────────
flag_savefig = 0

def plot_amplitude_recovery(df_rec, gamma_s, mu_str_for, mu_str_inv,
                             region=region1, flag_savefig=0):
    """
    3-panel PyGMT figure:
      (a) True amplitude  (b) Recovered amplitude  (c) Local recovery ratio
    """
    amp_max_mm = amp * m2mm   # 80 mm/yr

    fig = pygmt.Figure()

    panels = [
        ('true_amp_mm', 'viridis', [0, amp_max_mm, amp_max_mm/10],
         'Amplitude (mm/yr)', '(a) True amplitude', True),
        ('rec_amp_mm',  'viridis', [0, amp_max_mm, amp_max_mm/10],
         'Amplitude (mm/yr)', '(b) Recovered amplitude', True),
        ('ratio',       'viridis',   [0, 1.0, 0.1],
         'Local rec. ratio', '(c) Local rec. ratio', True),
    ]

    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, 
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", 
                     MAP_TITLE_OFFSET="-0.2c",MAP_SCALE_HEIGHT="3p", 
                     FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
 
        for i, (col, cmap, series, clabel, title, reverse) in enumerate(panels):
            
            with fig.set_panel(panel=i):
                fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title}"])
                
                scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
                mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
                fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

                pygmt.makecpt(cmap=cmap, series=series, reverse=reverse, background=True)

                # Fault patches — sort so high values render on top; dropna skips NaN ratio
                dp = df_rec[['lon','lat',col]].dropna().sort_values(col)
                print(dp[col].max())
                # print(dp['lon'])
                fig.plot(x=dp['lon'], y=dp['lat'], fill=dp[col],
                         style='c0.10c', cmap=True) #, pen='0.05p,gray50'

                # Plate contour & Trench
                # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], 
                #   region=region, spacing=(0.05, 0.05)) # no smoothing
                # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
                fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", 
                         style="f0.4i/0.075i+l+t", fill="dimgray")

                # GNSS stations
                fig.plot(x=df['lon'], y=df['lat'], style='s0.15c', fill='cyan', pen='0.25p,black')

                # Colorbar (below panel)
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", f'x+l{clabel}'],
                                 position="JBC+h+o0/0.5c")
                
                # fig.colorbar(frame=["af", "x+l@%1%S@%% magnitude", "y+l(mm)"], position="JBC+h+o0/0.5c")
                

    # Super-title
    label = (f"{meshname}  |  γ={gamma_s:.1e}  |  "
             f"fwd:{mu_str_for}  inv:{mu_str_inv}")
    fig.text(position='TC', text=label, font='9p,Helvetica', offset='0/0.3c',
             no_clip=True)

    fig.show()
    if flag_savefig:
        fname = (figpath + f"{meshname}_slip{pattern_option}_az_scalaramp"
                 f"_recovery_g{gamma_s:.1e}{mu_str_for}{mu_str_inv}.pdf")
        fig.savefig(fname, dpi=300)
        print("Saved:", fname)

plot_amplitude_recovery(df_rec, gamma_s, mu_str_for, mu_str_inv,
                        flag_savefig=flag_savefig)

In [ ]:
# ── Plot: recovery ratio across multiple γ values (4-panel or grid) ───────────
def plot_ratio_vs_gamma(recs, region=region1, flag_savefig=0):
    """
    One panel per γ, showing recovery ratio (rec_amp / amp).
    For flag_cf=True, recs is loaded from slip_recovery_*_cf_* via read_az_recovery,
    so only locked vertices are present and dropna() leaves them all.
    """
    gs_list = sorted(recs.keys())
    n = len(gs_list)
    ncols = min(n, 4)
    nrows = (n + ncols - 1) // ncols

    fig = pygmt.Figure()

    with fig.subplot(nrows=nrows, ncols=ncols,
                     figsize=(f'{5*ncols}c', f'{5.5*nrows}c'),
                     margins=['0.3c','0.5c']):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
            MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
     
        for i, gs in enumerate(gs_list):
            d = recs[gs]
            sm = d['true_amp_mm'] > 1.0
            r_min = np.nanmin(d.loc[sm,'ratio']); r_max = np.nanmax(d.loc[sm,'ratio'])
            title = f'(γ={gs:.1e})  ratio [{r_min:.2f}, {r_max:.2f}]'
            with fig.set_panel(panel=i):
                fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title}"])

                scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
                mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
                fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

                pygmt.makecpt(cmap='viridis', series=[0, 1.0, 0.1], reverse=True)

                dp = d[['lon','lat','ratio']].dropna().sort_values('ratio')
                fig.plot(x=dp['lon'], y=dp['lat'], fill=dp['ratio'],
                         style='c0.10c', cmap=True)
                fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", 'x+lLocal rec. ratio'],
                                 position="JBC+h+o0/0.5c")

    fig.show()
    if flag_savefig:
        fname = figpath + f"{meshname}_slip{pattern_option}_az_scalaramp_ratio_allgamma.pdf"
        fig.savefig(fname, dpi=300)
        print("Saved:", fname)

plot_ratio_vs_gamma(recs, flag_savefig=flag_savefig)

In [ ]:
# ── Het (slab-wedge) case: confirm no spots in cf recovery ────────────────────
df_rec_het = read_az_recovery(sw_str, inv_str, sw_str)
sm_het = df_rec_het['true_amp_mm'] > 1.0
print(f"Het | N vertices: {len(df_rec_het)}  |  stripe: {sm_het.sum()}")
print(f"Rec amp (stripe): [{df_rec_het.loc[sm_het,'rec_amp_mm'].min():.1f}, "
      f"{df_rec_het.loc[sm_het,'rec_amp_mm'].max():.1f}] mm/yr")
print(f"Recovery ratio (stripe): [{df_rec_het.loc[sm_het,'ratio'].min():.3f}, "
      f"{df_rec_het.loc[sm_het,'ratio'].max():.3f}]")

plot_amplitude_recovery(df_rec_het, gamma_s, sw_str, sw_str, flag_savefig=flag_savefig)

In [ ]:
# ── 3D (DeShon) case: confirm no spots in cf recovery ─────────────────────────
df_rec_3d = read_az_recovery(het3d_str, inv_str, het3d_str)
sm_3d = df_rec_3d['true_amp_mm'] > 1.0
print(f"3D | N vertices: {len(df_rec_3d)}  |  stripe: {sm_3d.sum()}")
print(f"Rec amp (stripe): [{df_rec_3d.loc[sm_3d,'rec_amp_mm'].min():.1f}, "
      f"{df_rec_3d.loc[sm_3d,'rec_amp_mm'].max():.1f}] mm/yr")      
print(f"Recovery ratio (stripe): [{df_rec_3d.loc[sm_3d,'ratio'].min():.3f}, "
      f"{df_rec_3d.loc[sm_3d,'ratio'].max():.3f}]")


plot_amplitude_recovery(df_rec_3d, gamma_s, het3d_str, het3d_str, flag_savefig=flag_savefig)

In [ ]:
# outFileName = f"Lcurvesynscaamp_azbd_rs{rho_s:.0e}_{meshname}_{slip_pattern}_{pattern_option}_{homo_str}_{homo_str}.txt"
# print(outFileName)
# gammas_CK_hom = gammas_available
# # gammas_CK_hom = gammas_CK_hom[::-1]
# misfitsCK_hom = pd.read_csv(resultpath + outFileName, sep=r'\s+',
#                     names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    

# import matplotlib.pyplot as plt

# # Plot L-curve
# fig, axes = plt.subplots(1, 1, figsize=(4,4), dpi=300)

# ax = axes
# ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
# ax.set_title("Synthetic locking ratio inversion", fontsize=9)
# ax.tick_params(labelsize=8)

# color_L = ['silver', 'firebrick', 'white']

# i_start, i_end = 1, -1
# i_start, i_end = 0, None

# ax.plot(misfitsCK_hom['m_mis'][i_start:i_end], misfitsCK_hom['d_mis'][i_start:i_end], 'o-', color='k', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"HOM; $\rho$={:.1e}".format(rho_s))
# # ax.text
# # ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 's-', color='red', markerfacecolor=color_L[2],
# #         linewidth=1.0, markersize=4, label=r"SW; SW")
# # ax.plot(misfitsCK_swhom['m_mis'], misfitsCK_swhom['d_mis'], 's--', color='red', markerfacecolor=color_L[0],
# #         linewidth=1.0, markersize=4, label=r"SW; HOM;")
# # ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 'D-', color='dodgerblue', markerfacecolor=color_L[2],
# #         linewidth=1.0, markersize=4, label=r"3D; 3D")
# # ax.plot(misfitsCK_3dhom['m_mis'], misfitsCK_3dhom['d_mis'], 'D--', color='dodgerblue', markerfacecolor=color_L[0],
# #         linewidth=1.0, markersize=4, label=r"3D; HOM;")

# for i, gamma in enumerate(gammas_CK_hom[i_start:i_end]):
#     ax.text(misfitsCK_hom['m_mis'][i_start:i_end].iloc[i]+0.25, misfitsCK_hom['d_mis'][i_start:i_end].iloc[i], f"{gamma:.0f}", fontsize=6, color='k')
# ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
# ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
# ax.set_box_aspect(1)
# ax.legend(fontsize=8)

In [ ]:
# ── Displacement helper functions (cf forward) ─────────────────────────────────

def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot
    return x_rot, y_rot

def read_cf_disp(df_sta, mu_str):
    outFileName = 'd_obs_cf_' + meshname + slip_str_gt + mu_str + '.txt'
    print(outFileName)
    u_true = pd.read_csv(resultpath + outFileName, sep=r'\s+', names=['x','y','z','ux','uy','uz'])
    u_true['lon'], u_true['lat'] = df_sta['lon'].values, df_sta['lat'].values
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot)
    u_true['mag'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()
    return u_true

def read_cf_disp_grid(sta_grid, mu_str, grid_spacing_deg=0.01):
    outFileName = 'd_obs_grid_cf_' + meshname + slip_str_gt + mu_str + str(grid_spacing_deg) + '.txt'
    print(outFileName)
    u_true = pd.read_csv(resultpath + outFileName, sep=r'\s+', names=['x','y','z','ux','uy','uz'])
    u_true['lon'], u_true['lat'] = sta_grid['lon'].values, sta_grid['lat'].values
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot)
    u_true['mag'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()
    return u_true

def build_disp_vector(u_true, scaleunit, error_e=None, error_n=None, error_v=None):
    df_obs_h_data = {
        'x': u_true['lon'],
        'y': u_true['lat'],
        'east_velocity': u_true['ux']*scaleunit,
        'north_velocity': u_true['uy']*scaleunit,
    }
    if error_e is not None and error_n is not None:
        df_obs_h_data['east_sigma'] = error_e*scaleunit
        df_obs_h_data['north_sigma'] = error_n*scaleunit
        df_obs_h_data['correlation_EN'] = 0.0
    df_obs_h = pd.DataFrame(data=df_obs_h_data)
    df_obs_v_data = {
        'x': u_true['lon'],
        'y': u_true['lat'],
        'east_velocity': 0.0,
        'north_velocity': u_true['uz']*scaleunit,
    }
    if error_v is not None:
        df_obs_v_data['east_sigma'] = error_v*scaleunit
        df_obs_v_data['north_sigma'] = error_v*scaleunit
        df_obs_v_data['correlation_EN'] = 0.0
    df_obs_v = pd.DataFrame(data=df_obs_v_data)
    return df_obs_h, df_obs_v

def build_disp_vector_grid(u_true, scaleunit, inc):
    unique_lons = u_true['lon'].unique()
    unique_lats = u_true['lat'].unique()
    n_lon = len(unique_lons)
    n_lat = len(unique_lats)
    print(f'Detected grid: {n_lat} x {n_lon}')
    ux_2d  = u_true['ux'].to_numpy().reshape(n_lat, n_lon)
    uy_2d  = u_true['uy'].to_numpy().reshape(n_lat, n_lon)
    lon_2d = u_true['lon'].to_numpy().reshape(n_lat, n_lon)
    lat_2d = u_true['lat'].to_numpy().reshape(n_lat, n_lon)
    ux_sub  = ux_2d[::inc, ::inc]
    uy_sub  = uy_2d[::inc, ::inc]
    lon_sub = lon_2d[::inc, ::inc]
    lat_sub = lat_2d[::inc, ::inc]
    print(f'Downsampled to: {ux_sub.shape[0]} x {ux_sub.shape[1]} = {ux_sub.size} points')
    df_obs_h_velo = pd.DataFrame(data={
        'x': lon_sub.flatten(),
        'y': lat_sub.flatten(),
        'east_velocity':  ux_sub.flatten() * scaleunit,
        'north_velocity': uy_sub.flatten() * scaleunit,
    })
    return df_obs_h_velo

def station_grid(regiongrid, grid_spacing_deg, depth_levels=[0]):
    lon_g = np.arange(regiongrid[0], regiongrid[1]+grid_spacing_deg, grid_spacing_deg)
    lat_g = np.arange(regiongrid[2], regiongrid[3]+grid_spacing_deg, grid_spacing_deg)
    LON, LAT = np.meshgrid(lon_g, lat_g)
    lon_2d, lat_2d = LON.flatten(), LAT.flatten()
    x_rot, y_rot = utp.LL2ckmd(lon_2d, lat_2d, lon0, lat0, rot)
    x_2d, y_2d = (x_rot - x0)/1e3, (y_rot - y0)/1e3
    n2d = len(x_2d)
    lon_d = np.tile(lon_2d, len(depth_levels))
    lat_d = np.tile(lat_2d, len(depth_levels))
    x_d   = np.tile(x_2d,  len(depth_levels))
    y_d   = np.tile(y_2d,  len(depth_levels))
    z_d   = np.repeat(depth_levels, n2d)
    return pd.DataFrame({'lon': lon_d, 'lat': lat_d, 'x': x_d, 'y': y_d, 'z': z_d})

# ── Build regular grid (z=0, surface only) ────────────────────────────────────
grid_spacing_deg = 0.01
sta_grid = station_grid([-87, -84, 8.6, 11.6], grid_spacing_deg)
print(f'Station grid: {len(sta_grid)} points')

# ── Plot function: copied verbatim from plot_true_slip_disp; only change is
#    panel 0 uses df_fault_cf['true_amp_mm'] instead of mtrue_s[col2plt]*amp*m2mm
def plot_cf_disp(df_fault_cf, scale_vec, df_obs_h, df_obs_v, region, struc_str_for, flag_savefig=False, u_true_grid=None):

    slipsybsz = 'c0.08c'
    colormap = 'viridis'

    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=('15c', '10c'), autolabel=False, frame=['a1f0.2', 'WSne'],
                    margins=['0.1c', '0.2c'], sharey=True):

        pygmt.config(MAP_FRAME_TYPE='plain', FONT_TITLE='8p,Helvetica-Bold,black', MAP_TITLE_OFFSET='-0.2c', MAP_SCALE_HEIGHT='3p',
                    FONT_ANNOT_PRIMARY='5p', FONT_ANNOT_SECONDARY='10p')

        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection='M?', frame=['a1f0.2', '+tTrue slip'])
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            fig.plot(x=df_fault_cf['lon'], y=df_fault_cf['lat'], style=slipsybsz, fill=df_fault_cf['true_amp_mm'], cmap=True)

            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.2
            mapscale_str = 'g' + str(scale_lon) + '/' + str(scale_lat) + 'c' + str(scale_lat) + '+w50k'
            fig.coast(borders=None, shorelines='0.25p,black', area_thresh=20, map_scale=mapscale_str)

            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit='-100/-5', annotation='20+f8p', pen='0.5p,darkgray')
            fig.plot(x=trench['lon'], y=trench['lat'], pen='1p,dimgray', style='f0.4i/0.075i+l+t', fill='dimgray')
            fig.plot(x=df['lon'], y=df['lat'], style='s0.15c', fill='cyan', pen='0.25p,black')
            with pygmt.config(FONT_ANNOT_PRIMARY='8p'):
                fig.colorbar(frame=['af', 'x+lSlip mag.', 'y+l(mm)'], position='JBC+h+o0/0.5c')

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection='M?', frame=['a1f0.2', '+tSynthetic horiz. disp.'], borders=None, shorelines='0.25p,black',
                      area_thresh=20, land='lightgray', water=None, resolution='h', map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit='-100/-5', annotation='20+f8p', pen='0.5p,darkgray')
            fig.plot(x=trench['lon'], y=trench['lat'], pen='1p,dimgray', style='f0.4i/0.075i+l+t', fill='dimgray')

            if u_true_grid is not None:
                griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_h'].max()*m2mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                fig.grdimage(grid=griduh, cmap=True, nan_transparent=True, interpolation='l')
                with pygmt.config(FONT_ANNOT_PRIMARY='8p'):
                    fig.colorbar(frame=['af', 'x+lHoriz. disp. mag.', 'y+l(mm)'], position='JBC+h+o0/0.5c')

            fig.velo(data=df_obs_h, pen='0.5p,black', uncertaintyfill=None, line='0.25p,black', spec='e'+str(0.5/scale_vec)+'/0.39',
                     vector='0.1c+a45+p1p,black+ea+gblack+h0')
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style='r2.6/2', fill='white', pen='0.5p,black', transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style='s0.15c', fill='cyan', pen='0.25p,black')
            lgdobs = pd.DataFrame(data={'x': region[0]+0.15, 'y': region[-2]+0.5, 'east_velocity': [1], 'north_velocity': [0],
                                  'east_sigma': 1/5, 'north_sigma': 1/5, 'correlation_EN': 0.0, })
            fig.velo(data=lgdobs, pen='0.5p,black', line='0.25p,black', spec='e0.5/0.39', vector='0.1c+a45+p1p,black+ea+gblack+h0')
            fig.text(text='Stations', x=region[0]+0.6, y=region[-2]+0.8, font='8p,Helvetica,black', justify='ML')
            fig.text(text='Obs H', x=region[0]+0.6, y=region[-2]+0.5, font='8p,Helvetica,black', justify='ML')
            fig.text(text=str(int(scale_vec))+'\u00b1'+str(int(scale_vec/5))+' mm', x=region[0]+0.1, y=region[-2]+0.2, font='8p,Helvetica,black', justify='ML')

        # Synthetic vertical displacement
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection='M?', frame=['a1f0.2', '+tSynthetic vert. disp.'], borders=None, shorelines='0.25p,black',
                      area_thresh=20, land='lightgray', water=None, resolution='h', map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit='-100/-5', annotation='20+f8p', pen='0.5p,darkgray')
            fig.plot(x=trench['lon'], y=trench['lat'], pen='1p,dimgray', style='f0.4i/0.075i+l+t', fill='dimgray')

            if u_true_grid is not None:
                griduz = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_v']*m2mm, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_v'].max()*m2mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                fig.grdimage(grid=griduz, cmap=True, nan_transparent=True, interpolation='l')
                with pygmt.config(FONT_ANNOT_PRIMARY='8p'):
                    fig.colorbar(frame=['af', 'x+lVert. disp. mag.', 'y+l(mm)'], position='JBC+h+o0/0.5c')

            fig.velo(data=df_obs_v, pen='0.5p,black', uncertaintyfill=None, line='0.25p,black', spec='e'+str(0.5/scale_vec)+'/0.39',
                     vector='0.1c+a45+p1p,black+ea+gblack+h0')
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style='r2.6/2', fill='white', pen='0.5p,black', transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style='s0.15c', fill='cyan', pen='0.25p,black')
            lgdobs = pd.DataFrame(data={'x': region[0]+0.4, 'y': region[-2]+0.3, 'east_velocity': [0], 'north_velocity': [1],
                                  'east_sigma': 1/5, 'north_sigma': 1/5, 'correlation_EN': 0.0, })
            fig.velo(data=lgdobs, pen='0.5p,black', line='0.25p,black', spec='e0.5/0.39', vector='0.1c+a45+p1p,black+ea+gblack+h0')
            fig.text(text='Stations', x=region[0]+0.6, y=region[-2]+0.8, font='8p,Helvetica,black', justify='ML')
            fig.text(text='Obs V', x=region[0]+0.6, y=region[-2]+0.5, font='8p,Helvetica,black', justify='ML')
            fig.text(text=str(int(scale_vec))+'\u00b1'+str(int(scale_vec/5))+' mm', x=region[0]+0.1, y=region[-2]+0.2, font='8p,Helvetica,black', justify='ML')

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + '_cf_truedisp.pdf')

def plot_cf_disp_grid(df_fault_cf, scale_vec, df_obs_h_grid, u_true_grid, region, struc_str_for,
                      uhcont_step=10, flag_savefig=False, df_obs_h_noerr=None):

    slipsybsz = 'c0.1c'
    # slipsybsz = 'c0.05c'
    # colormap = 'lajolla'
    colormap = 'viridis'

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=('15c', '10c'), autolabel=False, frame=['a1f0.2', 'WSne'],
                    margins=['0.1c', '0.2c'], sharey=True):

        pygmt.config(MAP_FRAME_TYPE='plain', FONT_TITLE='8p,Helvetica-Bold,black', MAP_TITLE_OFFSET='-0.15c', MAP_SCALE_HEIGHT='3p',
                    FONT_ANNOT_PRIMARY='5p', FONT_ANNOT_SECONDARY='10p')

        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection='M?', frame=['a1f0.2', '+tTrue slip'])
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)
            fig.plot(x=df_fault_cf['lon'], y=df_fault_cf['lat'], style=slipsybsz, fill=df_fault_cf['true_amp_mm'], cmap=True)   # no smoothing or interpolation

            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = 'g' + str(scale_lon) + '/' + str(scale_lat) + 'c' + str(scale_lat) + '+w50k'
            fig.coast(borders=None, shorelines='0.25p,black', area_thresh=20, map_scale=mapscale_str)

            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit='-100/-5', annotation='20+f8p', pen='0.5p,darkgray')
            fig.plot(x=trench['lon'], y=trench['lat'], pen='1p,dimgray', style='f0.4i/0.075i+l+t', fill='dimgray')
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style='t0.15c', fill='black', pen='0.25p,black')
            fig.plot(x=df['lon'], y=df['lat'], style='s0.15c', fill='cyan', pen='0.25p,black')

            with pygmt.config(FONT_ANNOT_PRIMARY='8p'):
                # fig.colorbar(frame=['af', 'x+lSlip mag.', 'y+l(mm)'])
                fig.colorbar(frame=['af', 'x+l@%1%S@%% magnitude', 'y+l(mm)'], position='JBC+h+o0/0.5c')

            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style='r1/0.6', fill='white', pen='0.1p,black', transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.3, style='s0.15c', fill='cyan', pen='0.25p,black')
            fig.text(text='Stations', x=region[0]+0.1, y=region[-2]+0.15, font='6p,Helvetica,black', justify='ML')

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection='M?', frame=['a1f0.2', '+tSynthetic disp.'])
            # use the vertical displacement as background color
            griduv = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            uv_max = u_true_grid['uz'].abs().max()*m2mm     # in mm
            print(uv_max)
            # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
            # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
            maxdisp = np.ceil(uv_max)
            # maxdisp = 36
            print(maxdisp)
            pygmt.makecpt(cmap='roma', series=[-maxdisp, maxdisp, maxdisp/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
            fig.grdimage(region=region, projection='M?', grid=griduv, cmap=True, nan_transparent=True, interpolation='l')
            with pygmt.config(FONT_ANNOT_PRIMARY='8p'):
                # fig.colorbar(frame=['af', 'x+lVert. disp.', 'y+l(mm)']) #
                # fig.colorbar(frame=['af', 'x+lU@-z@-', 'y+l(mm)']) #
                fig.colorbar(frame=['af', 'x+l@%1%U@%%@-z@-', 'y+l(mm)'], position='JBC+h+o0/0.5c')

            fig.coast(borders=None, shorelines='0.1p,black', area_thresh=20)
            # if struc_str_for == homo_str:
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit='-100/-5', annotation='20+f8p', pen='0.5p,darkgray')
            # fig.plot(x=trench['lon'], y=trench['lat'], pen='0.5p,dimgray', style='f0.4i/0.05i+l+t', fill='dimgray')
            fig.plot(x=trench['lon'], y=trench['lat'], pen='1p,dimgray', style='f0.4i/0.075i+l+t', fill='dimgray')

            # else:
            #     # plot the contour lines of horizontal displacement vector magnitude
            #     griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            #     # fig.grdcontour(grid=griduh, annotation='+f4p, white', pen='0.25p, white') #levels=2, 2
            #     fig.grdcontour(grid=griduh, levels=uhcont_step, annotation='+f4p, white', pen='0.25p, white') #, 2

            #plot the horizontal displacement vectors over the grid, error-free
            fig.velo(data=df_obs_h_grid, pen='0.5p,black', spec='e'+str(0.5/scale_vec)+'/0.39',
                     vector='0.1c+a45+p0.1p,black+ea+gwhite+h0')
            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style='r1/0.6', fill='white', pen='0.1p,black', transparency=30, )
            lgd = pd.DataFrame(data={'x': region[0]+0.15, 'y': region[-2]+0.3, 'east_velocity': [1], 'north_velocity': [0]})
            fig.velo(data=lgd, pen='0.5p,black', spec='e0.5/0.39', vector='0.1c+a45+p0.1p,black+ea+gwhite+h0')
            fig.text(text=str(int(scale_vec))+' mm', x=region[0]+0.1, y=region[-2]+0.15, font='6p,Helvetica,black', justify='ML')

        # direct comparison with error-free displacement vectors at real station locations
        if df_obs_h_noerr is not None:
            with fig.set_panel(panel=[0, 2]):
                fig.basemap(region=region, projection='M?', frame=['a1f0.2', '+tCompare at real stations'])
                #plot the horizontal displacement vectors over the grid, error-free
                fig.velo(data=df_obs_h_grid, pen='0.5p,black', spec='e'+str(0.5/scale_vec)+'/0.39',
                        vector='0.1c+a45+p0.1p,black+ea+gwhite+h0')
                #plot the horizontal displacement vectors over real station, error-free
                fig.velo(data=df_obs_h_noerr, pen='0.5p,magenta', spec='e'+str(0.5/scale_vec)+'/0.39',
                                    vector='0.1c+a45+p0.1p,magenta+ea+gmagenta+h0')
                # plot legends
                fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style='r1/0.8', fill='white', pen=None, transparency=30, )
                lgd = pd.DataFrame(data={'x': region[0]+0.15, 'y': region[-2]+0.3, 'east_velocity': [1], 'north_velocity': [0]})
                fig.velo(data=lgd, pen='0.5p,black', spec='e0.5/0.39', vector='0.1c+a45+p0.1p,black+ea+gwhite+h0')
                fig.text(text=str(int(scale_vec))+' mm', x=region[0]+0.1, y=region[-2]+0.15, font='6p,Helvetica,black', justify='ML')

    fig.show()

    if flag_savefig:
        figname = meshname + struc_str_for + str(slipmodel) + '_cf_truedisp_grid.pdf'
        fig.savefig(figpath + figname) #, crop=False


In [ ]:
# ── Hom cf displacement ────────────────────────────────────────────────────────
u_hom      = read_cf_disp(df, homo_str)
u_hom_grid = read_cf_disp_grid(sta_grid, homo_str, grid_spacing_deg)
df_obs_h_hom, df_obs_v_hom = build_disp_vector(u_hom, m2mm, error_e, error_n, error_v)
uh_max = np.hypot(df_obs_h_hom['east_velocity'].to_numpy(), df_obs_h_hom['north_velocity'].to_numpy()).max()
n = np.ceil(uh_max / 10)  # scale_vec=5*n keeps ratio < 2
scale_vec = np.round(5*n)
print(uh_max, scale_vec)
plot_cf_disp(df_fault, scale_vec, df_obs_h_hom, df_obs_v_hom, region, homo_str,
             flag_savefig=flag_savefig, 
            #  u_true_grid=u_hom_grid,
             )


In [ ]:
# ── Het (slab-wedge) cf displacement ─────────────────────────────────────────
u_het      = read_cf_disp(df, sw_str)
u_het_grid = read_cf_disp_grid(sta_grid, sw_str, grid_spacing_deg)
df_obs_h_het, df_obs_v_het = build_disp_vector(u_het, m2mm, error_e, error_n, error_v)
plot_cf_disp(df_fault, scale_vec, df_obs_h_het, df_obs_v_het, region1, sw_str,
             flag_savefig=flag_savefig, u_true_grid=u_het_grid)


In [ ]:
# ── 3D (DeShon) cf displacement ───────────────────────────────────────────────
u_3d      = read_cf_disp(df, het3d_str)
u_3d_grid = read_cf_disp_grid(sta_grid, het3d_str, grid_spacing_deg)
df_obs_h_3d, df_obs_v_3d = build_disp_vector(u_3d, m2mm, error_e, error_n, error_v)
plot_cf_disp(df_fault, scale_vec, df_obs_h_3d, df_obs_v_3d, region1, het3d_str,
             flag_savefig=flag_savefig, u_true_grid=u_3d_grid)


In [ ]:
# ── Hom cf displacement grid ───────────────────────────────────────────────────
uh_max = u_hom_grid['mag_h'].max()*m2mm
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec_grid = 5*n    # vector length are plotted relative to scale_vec*length in original unit, here mm
if slipmodel==2:
    scale_vec_grid = 45
elif slipmodel==3:
    scale_vec_grid = 15
elif slipmodel==4:
    scale_vec_grid = 20
print(uh_max, scale_vec_grid)
df_obs_h_hom_grid = build_disp_vector_grid(u_hom_grid, m2mm, inc=20)
plot_cf_disp_grid(df_fault, scale_vec_grid, df_obs_h_hom_grid, u_hom_grid, region1, homo_str,
                  flag_savefig=flag_savefig)


In [ ]:
# ── Het (slab-wedge) cf displacement grid ────────────────────────────────────
df_obs_h_het_grid = build_disp_vector_grid(u_het_grid, m2mm, inc=20)
plot_cf_disp_grid(df_fault, scale_vec_grid, df_obs_h_het_grid, u_het_grid, region1, sw_str,
                  flag_savefig=flag_savefig)


In [ ]:
# ── 3D (DeShon) cf displacement grid ─────────────────────────────────────────
df_obs_h_3d_grid = build_disp_vector_grid(u_3d_grid, m2mm, inc=20)
plot_cf_disp_grid(df_fault, scale_vec_grid, df_obs_h_3d_grid, u_3d_grid, region1, het3d_str,
                  flag_savefig=flag_savefig)
